# Unit 2 | NLP 01 — Tokenization & Text Preprocessing

Before we can model language we need to answer: **what is a token?** This notebook walks the full preprocessing pipeline:

1. **Tokenization** — splitting text into words
2. **Normalization** — lowercasing, removing punctuation
3. **Stopwords** — filtering high-frequency, low-information words
4. **Stemming & Lemmatization** — reducing words to root forms
5. **N-grams** — capturing multi-word expressions
6. **Vocabulary-size experiment** — measuring the cumulative effect of each step

**Dataset:** sklearn 20 Newsgroups (built-in, no download needed)

In [1]:
import re
import nltk
import numpy as np
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
plt.style.use('dark_background')

for resource in ['punkt', 'punkt_tab', 'stopwords', 'wordnet']:
    nltk.download(resource, quiet=True)

In [2]:
# ── CONFIGURATION ────────────────────────────────────────────────────────────
# Change CATEGORIES to any valid 20newsgroups group names.
# Valid names: fetch_20newsgroups().target_names

CATEGORIES = ['sci.space', 'comp.graphics', 'rec.sport.hockey', 'talk.religion.misc']
RANDOM_STATE = 42
# ─────────────────────────────────────────────────────────────────────────────

## What Is NLP?

**Natural Language Processing (NLP)** is the branch of machine learning that deals with text and language data.

### The preprocessing pipeline

```
Raw Text
   │
   ▼  Tokenization     — split into individual tokens
   │
   ▼  Normalization    — lowercase, strip punctuation
   │
   ▼  Stopword Removal — drop high-frequency, low-signal words
   │
   ▼  Stem / Lemmatize — reduce to root forms  (optional)
   │
   ▼  N-gram Extraction — capture phrases       (optional)
   │
   ▼
Numerical Representation  ←── Notebook 2
```

In [3]:
# Load 20 Newsgroups
# remove= strips headers, footers, quoted replies — leaves the actual post body
data = fetch_20newsgroups(
    subset='train',
    categories=CATEGORIES,
    remove=('headers', 'footers', 'quotes'),
    random_state=RANDOM_STATE
)

print(f"Loaded {len(data.data):,} documents across {len(data.target_names)} categories")
print(f"Categories: {data.target_names}")
print(f"\n{'='*60}")
cat_label = data.target_names[data.target[0]]
print(f"Sample document (category: {cat_label}):")
print('='*60)
print(data.data[0][:500])

Loaded 2,154 documents across 4 categories
Categories: ['comp.graphics', 'rec.sport.hockey', 'sci.space', 'talk.religion.misc']

Sample document (category: rec.sport.hockey):
According to THE FAN here in T.O., Ottawa has won the Daigle e
sweepstakes.  They didn't mention why, but San Jose had more goals
than the Sen-sens, so I have a hunch this is why Ottawa would pick
first.....


# 0. Motivation

In [36]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    lowercase=True,
    strip_accents='unicode',
    token_pattern=r"\b[\w\-']+\b",
    stop_words='english',
    min_df=10,      # must appear in at least 5 documents
    max_df=0.75,   # ignore terms appearing in more than 95% of documents
    max_features=100,   # keeps the 1000 most frequent terms across the corpus
)

X = vectorizer.fit_transform(data.data)

df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
sums = df.sum(axis=0).sort_values(ascending=False)
df = df[sums.index]  # reorder columns by frequency
df['_target_'] = [data.target_names[t] for t in data.target]

df

,1,0,2,3,4,space,5,7,6,8,like,just,don't,9,edu,know,think,10,time,people,image,team,good,new,does,data,it's,use,game,graphics,25,i'm,nasa,program,hockey,11,year,play,16,55,available,did,way,12,15,14,20,18,make,god,...,say,point,period,years,season,software,information,13,images,said,24,file,c,games,right,need,better,power,want,launch,17,x,used,jpeg,jesus,files,got,using,ftp,center,19,really,points,nhl,look,o,world,21,la,work,read,thanks,things,23,line,players,g,can't,great,_target_
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,rec.sport.hockey
1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,comp.graphics
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,rec.sport.hockey
3,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,6,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,sci.space
4,0,0,0,0,0,2,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,sci.space
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2149,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,talk.religion.misc
2150,1,0,3,0,1,0,0,0,0,0,0,0,0,0,2,0,1,0,2,0,0,0,0,0,0,0,1,1,0,1,0,2,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,comp.graphics
2151,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,2,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,2,1,0,0,0,0,1,0,0,comp.graphics
2152,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,talk.religion.misc


## 1. Tokenization

**Tokenization** splits a string into tokens. The simplest approach is `str.split()`. NLTK's `word_tokenize` is smarter — it handles punctuation, contractions, and edge cases correctly.

In [22]:
from nltk.tokenize import word_tokenize, TweetTokenizer, MWETokenizer

sample = data.data[0][:500]

# --- tokenize ---
split_tokens = sample.split()       # sep=None splits on any whitespace

nltk_tokens = word_tokenize(
    sample,
    language='english'
)

tweet_tokenizer = TweetTokenizer(
    preserve_case=True,
    strip_handles=False,
    reduce_len=False
)
tweet_tokens    = tweet_tokenizer.tokenize(sample)

mwe_tokenizer = MWETokenizer(
    mwes=[
        ("New", "York"),
        ("machine", "learning")
    ],
    separator='_'
)
mwe_tokens    = mwe_tokenizer.tokenize(word_tokenize(sample))

vectorizer   = CountVectorizer(
    lowercase=False,
    strip_accents='unicode',
    token_pattern=r"\b\S+\b"
)
vectorizer.fit([sample])
sklearn_tokens = vectorizer.build_analyzer()(sample)

# --- align lengths ---
all_tokens = [split_tokens, nltk_tokens, tweet_tokens, mwe_tokens, sklearn_tokens]
max_len    = max(len(t) for t in all_tokens)

def pad(tokens):
    return tokens + [''] * (max_len - len(tokens))

df_tokens = pd.DataFrame({
    "str.split" : pad(split_tokens),
    "nltk"      : pad(nltk_tokens),
    "tweet"     : pad(tweet_tokens),
    "mwe"       : pad(mwe_tokens),
    "sklearn"   : pad(sklearn_tokens),
})

df_tokens

,str.split,nltk,tweet,mwe,sklearn
0,According,According,According,According,According
1,to,to,to,to,to
2,THE,THE,THE,THE,THE
3,FAN,FAN,FAN,FAN,FAN
4,here,here,here,here,here
5,in,in,in,in,in
6,"T.O.,",T.O.,T,T.O.,T.O
7,Ottawa,",",.,",",Ottawa
8,has,Ottawa,O,Ottawa,has
9,won,has,.,has,won


## 2. Normalization

After tokenizing we **lowercase** and **strip punctuation** so `NASA`, `nasa`, and `Nasa` all map to the same token.

In [26]:
sample_raw = data.data[0][:500]
print(f"Original sample (length: {len(sample_raw)}):\n")
print(sample_raw)

Original sample (length: 207):

According to THE FAN here in T.O., Ottawa has won the Daigle e
sweepstakes.  They didn't mention why, but San Jose had more goals
than the Sen-sens, so I have a hunch this is why Ottawa would pick
first.....


In [28]:
sample_clean = re.sub(r'\s+', ' ', sample_raw).strip()
print(f"Cleaned sample (length: {len(sample_clean)}):\n")
print(sample_clean)

Cleaned sample (length: 206):

According to THE FAN here in T.O., Ottawa has won the Daigle e sweepstakes. They didn't mention why, but San Jose had more goals than the Sen-sens, so I have a hunch this is why Ottawa would pick first.....


## 3. Stopwords

**Stopwords** are common words (*the, is, at, which, on …*) that appear in nearly every document and carry little discriminative information. Removing them shrinks the vocabulary and often helps models focus on meaningful words.

In [ ]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

print(f"NLTK has {len(stop_words)} English stopwords. First 30 (alphabetical):")
print(sorted(stop_words)[:30])

In [ ]:
# Measure vocabulary size before and after stopword removal
raw_vocab, filtered_vocab = set(), set()
raw_tokens, filt_tokens = [], []

for doc in data.data[:300]:
    tokens = re.sub(r'[^a-z\s]', '', doc.lower()).split()
    raw_vocab.update(tokens)
    raw_tokens.extend(tokens)
    kept = [t for t in tokens if t not in stop_words and len(t) > 1]
    filtered_vocab.update(kept)
    filt_tokens.extend(kept)

pct = 100 * (len(raw_vocab) - len(filtered_vocab)) / len(raw_vocab)
print(f"Vocabulary BEFORE stopword removal: {len(raw_vocab):>6,}")
print(f"Vocabulary AFTER  stopword removal: {len(filtered_vocab):>6,}")
print(f"Reduction: {pct:.1f}%")

print("\nTop 10 tokens BEFORE removal:")
print([w for w, _ in Counter(raw_tokens).most_common(10)])

print("\nTop 10 tokens AFTER removal:")
print([w for w, _ in Counter(filt_tokens).most_common(10)])

## 4. Stemming vs. Lemmatization

Both collapse word variants to a common root, but with different trade-offs:

| | PorterStemmer | WordNetLemmatizer |
|---|---|---|
| Speed | Fast | Slower (uses dictionary) |
| Output | May not be a real word | Always a real word |
| `running` → | `run` | `running` (needs POS tag for `run`) |
| `better` → | `better` | `good` |
| `studies` → | `studi` | `study` |

For bag-of-words pipelines either works. Lemmatization is more interpretable.

In [23]:
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

stemmer    = PorterStemmer()
lemmatizer = WordNetLemmatizer()

words = ['running', 'runs', 'ran', 'runner',
         'better', 'good', 'easily',
         'studies', 'studying', 'studied',
         'organization', 'organizing']

print(f"{'Word':<16} {'Stemmed':<16} {'Lemmatized'}")
print('-' * 48)
for w in words:
    print(f"{w:<16} {stemmer.stem(w):<16} {lemmatizer.lemmatize(w)}")

Word             Stemmed          Lemmatized
------------------------------------------------
running          run              running
runs             run              run
ran              ran              ran
runner           runner           runner
better           better           better
good             good             good
easily           easili           easily
studies          studi            study
studying         studi            studying
studied          studi            studied
organization     organ            organization
organizing       organ            organizing


## 5. N-grams

A **unigram** is one word. A **bigram** is two consecutive words. An **n-gram** is any sequence of n consecutive words.

N-grams capture local context that single-word models miss:
- `"not good"` as a bigram ≠ `"not"` + `"good"` as separate unigrams.

In [ ]:
def make_ngrams(tokens, n):
    return list(zip(*[tokens[i:] for i in range(n)]))

demo = ['the', 'space', 'shuttle', 'launched', 'from', 'cape', 'canaveral']
print("Tokens:", demo)
print("Bigrams: ", make_ngrams(demo, 2))
print("Trigrams:", make_ngrams(demo, 3))

In [ ]:
# Visualize top 10 bigrams per category
n_cats = len(CATEGORIES)
fig, axes = plt.subplots(1, n_cats, figsize=(5 * n_cats, 5))
if n_cats == 1:
    axes = [axes]

for ax, cat in zip(axes, CATEGORIES):
    cat_idx  = data.target_names.index(cat)
    cat_docs = [doc for doc, lbl in zip(data.data, data.target) if lbl == cat_idx]

    bigrams = []
    for doc in cat_docs[:120]:
        tokens = re.sub(r'[^a-z\s]', '', doc.lower()).split()
        tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
        bigrams.extend(' '.join(bg) for bg in make_ngrams(tokens, 2))

    top = Counter(bigrams).most_common(10)
    if top:
        labels, counts = zip(*top)
        ax.barh(list(labels)[::-1], list(counts)[::-1])
    ax.set_title(cat.split('.')[-1].replace('_', ' ').title(), fontsize=11)
    ax.set_xlabel('Count')

plt.suptitle('Top 10 Bigrams per Category', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 6. How Preprocessing Choices Affect Vocabulary Size

In [ ]:
configs = {
    'Raw split':     (False, False, False),
    'Lowercase':     (True,  False, False),
    '+ Stopwords':   (True,  True,  False),
    '+ Stemming':    (True,  True,  True),
}

vocab_sizes = {}
for label, (do_lower, do_stop, do_stem) in configs.items():
    vocab = set()
    stemmer = PorterStemmer()
    pattern = r'[^a-z\s]' if do_lower else r'[^a-zA-Z\s]'
    for doc in data.data[:400]:
        text   = doc.lower() if do_lower else doc
        tokens = re.sub(pattern, '', text).split()
        if do_stop:
            tokens = [t for t in tokens if t not in stop_words]
        if do_stem:
            tokens = [stemmer.stem(t) for t in tokens]
        vocab.update(tokens)
    vocab_sizes[label] = len(vocab)

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#5b8dd9', '#5b8dd9', '#5b8dd9', '#d95b5b']
bars   = ax.bar(list(vocab_sizes.keys()), list(vocab_sizes.values()), color=colors)
ax.set_ylabel('Unique Tokens (Vocabulary Size)')
ax.set_title('Cumulative Effect of Preprocessing on Vocabulary Size')
for bar, val in zip(bars, vocab_sizes.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
            f'{val:,}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

print("Takeaway: aggressive preprocessing reduces vocab by ~60-70%,")
print("which means fewer features and faster training.")

## Summary

| Step | What It Does | When to Skip |
|---|---|---|
| Tokenization | Splits text into tokens | Never |
| Normalization | Lowercases, strips punctuation | Rarely |
| Stopword removal | Filters common words | Sentiment analysis |
| Stemming | Fast approximate root reduction | Interpretability matters |
| Lemmatization | Dictionary root reduction | Speed matters |
| N-grams | Captures word sequences | Very high-dimensional data |

**Next →** Notebook 2 converts these token lists into numeric feature vectors.